In [ ]:
import pandas as pd
from sqlalchemy import create_engine
import pymysql

# --- STEP 1: LOAD DATA (EXTRACT) ---
print("Sedang membaca data Amazon.")

df = pd.read_csv('Amazon Sale Report.csv', low_memory=False)

# --- STEP 2: DATA CLEANING (TRANSFORM) ---
print("Sedang membersihkan data (Data Cleaning)...")

# 1. Merapihkan Nama Kolom (Buat jadi snake_case: huruf kecil & pake underscore)
df.columns = df.columns.str.replace(' ', '_').str.replace('-', '_').str.lower()

# 2. Ubah format Tanggal (Date) jadi format Datetime asli
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# 3. Menangani Data Kosong (Missing Values)
# mengisi kolom Amount yang kosong dengan 0
df['amount'] = df['amount'].fillna(0)
# mengisi kolom kategori & kota yang kosong dengan 'Unknown'
df['category'] = df['category'].fillna('Unknown')
df['ship_city'] = df['ship_city'].fillna('Unknown')

# 4. Hapus kolom 'index' bawaan CSV agar tidak ganda dengan index SQL
if 'index' in df.columns:
    df = df.drop(columns=['index'])

# --- STEP 3: SEND TO SQL (LOAD) ---
print("Mengirim data ke SQL")

# Koneksi ke XAMPP (Username: root, Password: kosong, Host: localhost, DB: amazon_analysis)
try:
    engine = create_engine('mysql+pymysql://root:@localhost/amazon_analysis')
    
    # Kirim data ke tabel bernama 'amazon_sales'
    # if_exists='replace' artinya kalau tabel sudah ada, akan ditimpa yang baru
    df.to_sql('amazon_sales', con=engine, index=False, if_exists='replace')
    
    print("\n--- STATUS: BERHASIL! ---")
    print("Data Amazon sudah tersimpan di database MySQL.")
    print(f"Total baris yang berhasil diproses: {len(df)}")

except Exception as e:
    print(f"\nWaduh, ada error pas konek ke SQL: {e}")
    print("Pastikan MySQL di XAMPP sudah dinyalakan ya!")

# --- STEP 4: INTIP HASILNYA ---
print("\nPreview Data Terbersih:")
print(df.head())

Sedang membaca data Amazon.
Sedang membersihkan data (Data Cleaning)...
Mengirim data ke SQL


C:\Users\deaal\AppData\Local\Temp\ipykernel_4360\2239791863.py:18: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['date'] = pd.to_datetime(df['date'], errors='coerce')



Waduh, ada error pas konek ke SQL: (pymysql.err.OperationalError) (2003, "Can't connect to MySQL server on 'localhost' ([WinError 10061] No connection could be made because the target machine actively refused it)")
(Background on this error at: https://sqlalche.me/e/20/e3q8)
Pastikan MySQL di XAMPP sudah dinyalakan ya!

Preview Data Terbersih:
              order_id       date                        status fulfilment  \
0  405-8078784-5731545 2022-04-30                     Cancelled   Merchant   
1  171-9198151-1101146 2022-04-30  Shipped - Delivered to Buyer   Merchant   
2  404-0687676-7273146 2022-04-30                       Shipped     Amazon   
3  403-9615377-8133951 2022-04-30                     Cancelled   Merchant   
4  407-1069790-7240320 2022-04-30                       Shipped     Amazon   

  sales_channel_ ship_service_level    style              sku       category  \
0      Amazon.in           Standard   SET389   SET389-KR-NP-S            Set   
1      Amazon.in        